In [0]:
# Stop any running streaming queries
for s in spark.streams.active:
    s.stop()

# Delete the checkpoint so Spark forgets previous file-processing progress
dbutils.fs.rm(CKPT, True)

# Delete the Delta table so we start with zero rows
spark.sql(f"DROP TABLE IF EXISTS {TABLE}")

print("Clean reset completed.")

Clean reset completed.


In [0]:
# Simulate the first two batches of incoming orders.
# Each file contains 5 JSON records.

drop_file("001", rows=5)
drop_file("002", rows=5)

Wrote 274 bytes.
Dropped orders_001.json (5 rows)
Wrote 270 bytes.
Dropped orders_002.json (5 rows)


In [0]:
# Confirm that the two files were actually created.

display(dbutils.fs.ls(SOURCE))

[]

In [0]:
# ============================================================
# CLIP 5 — SETUP 4: RESET
# ============================================================
# Hint:
# Run this ONLY when you want a completely clean experiment.
#
# It removes:
#   - previous checkpoint files
#   - previous Delta table
#
# It does NOT remove the Python variables such as:
#   TABLE
#   CKPT
#   SOURCE
#
# We will NOT run this between incremental runs.
# ============================================================

for s in spark.streams.active:
    s.stop()

dbutils.fs.rm(CKPT, True)

spark.sql(f"DROP TABLE IF EXISTS {TABLE}")

print("Reset complete.")

In [0]:
# Stop any running streaming queries
for s in spark.streams.active:
    s.stop()

# Delete ALL files from the source folder
dbutils.fs.rm(SOURCE, True)

# Recreate the empty source folder
dbutils.fs.mkdirs(SOURCE)

# Delete the checkpoint so Auto Loader starts completely fresh
dbutils.fs.rm(CKPT, True)

# Delete the Delta table
spark.sql(f"DROP TABLE IF EXISTS {TABLE}")

print("Everything reset. Ready to start from zero.")

Everything reset. Ready to start from zero.


In [0]:
# This should show an empty folder.

display(dbutils.fs.ls(SOURCE))

[]

In [0]:
# ============================================================
# CLIP 5 — SETUP 1
# ============================================================
# Hint:
# We create:
#   1. Schema
#   2. Volume
#   3. Source folder
#   4. Checkpoint folder
#   5. Target Delta table
#
# This is a NEW practice area.
# It does NOT touch our previous watermark practice.
# ============================================================

CATALOG = "retail"
SCHEMA  = "sales"

spark.sql(f"CREATE SCHEMA IF NOT EXISTS {CATALOG}.{SCHEMA}")

spark.sql(f"""
    CREATE VOLUME IF NOT EXISTS {CATALOG}.{SCHEMA}.trigger_demo
""")

BASE   = f"/Volumes/{CATALOG}/{SCHEMA}/trigger_demo"
SOURCE = f"{BASE}/source"
CKPT   = f"{BASE}/checkpoints"
TABLE  = f"{CATALOG}.{SCHEMA}.orders_bronze"

print("Source folder:", SOURCE)
print("Checkpoints  :", CKPT)
print("Target table :", TABLE)

Source folder: /Volumes/retail/sales/trigger_demo/source
Checkpoints  : /Volumes/retail/sales/trigger_demo/checkpoints
Target table : retail.sales.orders_bronze


In [0]:
# ============================================================
# CLIP 5 — SETUP 2
# ============================================================
# Hint:
# drop_file() simulates new files arriving in our source folder.
#
# For example:
#
# drop_file("001")
#
# means:
# "A new batch/file of orders has arrived."
# ============================================================

import json
import random

def drop_file(batch_name, rows=5):
    """Write one small JSON file into the source folder."""

    regions = ["north", "south", "east", "west"]

    lines = [
        json.dumps({
            "order_id": random.randint(1000, 9999),
            "region": random.choice(regions),
            "amount": round(random.uniform(10, 200), 2)
        })
        for _ in range(rows)
    ]

    dbutils.fs.put(
        f"{SOURCE}/orders_{batch_name}.json",
        "\n".join(lines),
        overwrite=True
    )

    print(f"Dropped orders_{batch_name}.json ({rows} rows)")

In [0]:
# ============================================================
# CLIP 5 — SETUP 3
# ============================================================
# Hint:
# Auto Loader watches SOURCE for new JSON files.
#
# IMPORTANT:
# readStream only defines the streaming DataFrame.
# Nothing is actually processed yet.
#
# Processing starts only when we call writeStream.
# ============================================================

ORDER_SCHEMA = "order_id INT, region STRING, amount DOUBLE"

orders = (
    spark.readStream
    .format("cloudFiles")
    .option("cloudFiles.format", "json")
    .schema(ORDER_SCHEMA)
    .load(SOURCE)
)

print("Streaming DataFrame ready.")
print("Is streaming?", orders.isStreaming)

Streaming DataFrame ready.
Is streaming? True


In [0]:
drop_file("001", rows=5)

Wrote 275 bytes.
Dropped orders_001.json (5 rows)


In [0]:
# Step 1 — Serverless-compatible trigger
# availableNow processes the files currently available, then stops automatically.

q = (
    orders.writeStream
    .format("delta")
    .outputMode("append")
    .option("checkpointLocation", f"{CKPT}/bronze")
    .trigger(availableNow=True)
    .toTable(TABLE)
)

# Wait until this batch finishes
q.awaitTermination()

print("First batch finished.")

First batch finished.


In [0]:
# Check orders_001.json directly.
display(
    spark.read.json(f"{SOURCE}/orders_001.json")
)

amount,order_id,region
182.31,6561,west
177.59,3269,north
199.45,3948,south
131.69,2063,east
63.2,2681,south


In [0]:
print(SOURCE)
display(dbutils.fs.ls(SOURCE))

/Volumes/retail/sales/trigger_demo/source


path,name,size,modificationTime
dbfs:/Volumes/retail/sales/trigger_demo/source/orders_001.json,orders_001.json,275,1791014511000


In [0]:

display(
    spark.read.table(TABLE)
    .groupBy("region")
    .count()
    .orderBy("region")
)

region,count
east,1
north,1
south,2
west,1


In [0]:
# Add NEW data after the first streaming run has finished.
# Use a new filename so Auto Loader can recognize it as a new file.

drop_file("002", rows=5)

Wrote 274 bytes.
Dropped orders_002.json (5 rows)


In [0]:
# Run the same streaming query again.
# IMPORTANT: We use the SAME checkpoint.
# The checkpoint remembers which source files were already processed.

q = (
    orders.writeStream
    .format("delta")
    .outputMode("append")
    .option("checkpointLocation", f"{CKPT}/bronze")
    .trigger(availableNow=True)
    .toTable(TABLE)
)

q.awaitTermination()

print("Second batch finished.")

Second batch finished.


In [0]:
print(SOURCE)
display(dbutils.fs.ls(SOURCE))

/Volumes/retail/sales/trigger_demo/source


path,name,size,modificationTime
dbfs:/Volumes/retail/sales/trigger_demo/source/orders_001.json,orders_001.json,275,1791014511000
dbfs:/Volumes/retail/sales/trigger_demo/source/orders_002.json,orders_002.json,274,1791014555000


In [0]:
# Check orders_002.json directly.
display(
    spark.read.json(f"{SOURCE}/orders_002.json")
)

amount,order_id,region
185.49,9563,north
69.41,8384,east
28.31,8999,south
177.62,7418,north
90.81,8707,east


In [0]:
# Check how many rows are now in the Delta table.
# We should see the original rows + the 5 new rows from orders_003.json.

display(
    spark.read.table(TABLE)
    .groupBy("region")
    .count()
    .orderBy("region")
)

region,count
east,3
north,3
south,3
west,1


In [0]:
# Simulate another arrival of data.
# This file did NOT exist during the previous streaming run.

drop_file("003", rows=5)

Wrote 270 bytes.
Dropped orders_003.json (5 rows)


In [0]:
# Same table + same checkpoint.
# Auto Loader should skip files it has already processed
# and process the newly arrived orders_004.json.

q = (
    orders.writeStream
    .format("delta")
    .outputMode("append")
    .option("checkpointLocation", f"{CKPT}/bronze")
    .trigger(availableNow=True)
    .toTable(TABLE)
)

q.awaitTermination()

print("Third batch finished.")

Third batch finished.


In [0]:
# Check orders_002.json directly.
display(
    spark.read.json(f"{SOURCE}/orders_003.json")
)

amount,order_id,region
39.81,6804,east
56.53,7747,west
57.57,3562,north
16.3,8328,east
31.61,2719,north


In [0]:
# Verify the cumulative data in the Delta table.

display(
    spark.read.table(TABLE)
    .groupBy("region")
    .count()
    .orderBy("region")
)

region,count
east,5
north,5
south,3
west,2


In [0]:
# Show every row currently stored in the Delta table.
# This lets us see exactly which orders were processed.

display(
    spark.read.table(TABLE)
    .orderBy("order_id")
)

order_id,region,amount
2063,east,131.69
2681,south,63.2
2719,north,31.61
3269,north,177.59
3562,north,57.57
3948,south,199.45
6561,west,182.31
6804,east,39.81
7418,north,177.62
7747,west,56.53
